In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

import pandas as pd

from baselines_finetune import BaseLineTrainer

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Baselines train + eval

This notebook uses `baselines_finetune.py`.

Flow for each model:
1. train best params with `opt_search_*`
2. evaluate on test with `objective_*(eval=True)`

In [2]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_baseline_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 50

In [3]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [4]:
trainer = BaseLineTrainer(
    data_config=data_config,
    metric=metric,
    auction_mode=auction_mode,
    base_params_subfolder=best_params_subfolder,
    random_state=42,
)

In [5]:
# models = ["broi"]  # e.g. ["linear", "tapid", "mpid", "broi"]
models = ["tapid", "mpid", "broi"]

rows = []
for model_name in models:
    print(f"=== {model_name.upper()} | metric={metric} | auction_mode={auction_mode} ===")
    getattr(trainer, f"opt_search_{model_name}")(n_trials=n_trials)
    score = getattr(trainer, f"objective_{model_name}")(None, eval=True)
    cpc, rmse, scr = score[0], score[1], score[2]
    rows.append({"model": model_name, "CPC_REL": cpc, "RMSE": rmse, "SCR": scr})

results_df = pd.DataFrame(rows)
sort_col = {"SCR": "SCR", "RMSE": "RMSE", "CPC_REL": "CPC_REL"}[metric]
results_df.sort_values(sort_col, ascending=(metric != "SCR")).reset_index(drop=True)

[I 2026-04-28 12:48:57,588] A new study created in memory with name: no-name-bf557092-73df-4dfb-bf68-0914bb339faf


=== TAPID | metric=SCR | auction_mode=FPA ===


[I 2026-04-28 12:49:38,020] Trial 0 finished with value: 3069.5665683207426 and parameters: {'k_p1': 0.003148911647956862, 'k_i1': 0.6351221010640696, 'k_d1': 0.08471801418819976, 'coef': 0.37261932455399976}. Best is trial 0 with value: 3069.5665683207426.


CPC_REL: 11.65475727462579, rmse: 7.479583509587028, SCR: 3069.5665683207426


[I 2026-04-28 12:50:14,104] Trial 1 finished with value: 1876.7011407300224 and parameters: {'k_p1': 0.00042079886696066364, 'k_i1': 0.0004207053950287938, 'k_d1': 0.0001707396743152812, 'coef': 0.670721300885889}. Best is trial 0 with value: 3069.5665683207426.


...

[I 2026-04-28 16:39:11,272] Trial 48 finished with value: 5335.313001746926 and parameters: {'ro': 0.07726942088288406, 'v_bar': 43.27410301233733}. Best is trial 0 with value: 5335.313001746926.


CPC_REL: 5366.064976843506, rmse: 1.2283139294795586, SCR: 5335.313001746926


[I 2026-04-28 16:40:10,952] Trial 49 finished with value: 5335.313001746926 and parameters: {'ro': 23.111640859271496, 'v_bar': 22.124040091582167}. Best is trial 0 with value: 5335.313001746926.


CPC_REL: 5366.064976843506, rmse: 1.2283139294795586, SCR: 5335.313001746926
Best trial:
  Value: 5335.313001746926
  Params: 
    ro: 0.4082331665966555
    v_bar: 122.7582671086234
CPC_REL: 5906.638103233931, rmse: 1.2297089006198547, SCR: 4599.856980283712


,model,CPC_REL,RMSE,SCR
0,mpid,468.089081,1.408079,14673.570616
1,tapid,10.940620,7.722339,8623.413930
2,broi,5906.638103,1.229709,4599.856980


### Notes

- `objective_*(eval=True)` loads params from:
  `best_params/{best_params_subfolder}/{model}_{metric}_{auction_mode}.pkl`
- This notebook currently runs: `linear`, `tapid`, `mpid`, `mystique`.
- `broi` is not included in this run block.

### Скоры val / test по сохранённым `.pkl`

Ниже: все `*{metric}_{auction}.pkl` из папки `trainer` + `autobidder_check` на каждом из `val`, `test`, если ключ есть в `data_config`.

In [6]:
from pathlib import Path
import pickle

from example_notebooks.experiments.adapters.baseline_adapter import (
    _MODEL_TO_BIDDER,
    _build_bidder_eval_params,
)
from simulator.validation.check_results import autobidder_check

params_dir = Path(trainer.get_params_path("linear")).parent
suffix = f"_{trainer.metric.lower()}_{trainer.auction_mode}.pkl"

rows = []
for split in ("val", "test"):
    if split not in data_config:
        continue
    sp = data_config[split]
    for path in sorted(params_dir.glob(f"*{suffix}")):
        stem = path.name[: -len(suffix)]
        d = pickle.loads(path.read_bytes())
        bidder = _MODEL_TO_BIDDER[stem]
        p = _build_bidder_eval_params(stem, d)
        sc = autobidder_check(
            bidder=bidder,
            params={"input_campaigns": sp["campaigns_path"], "input_stats": sp["stats_path"], **p},
            auction_mode=trainer.auction_mode,
        )["score"]
        rows.append({"split": split, "model": stem, "CPC_REL": sc[0], "RMSE": sc[1], "SCR": sc[2]})

baseline_scores_df = pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)
baseline_scores_df

,split,model,CPC_REL,RMSE,SCR
0,test,broi,5906.638103,1.229709,4599.856980
1,test,linear,421.338042,1.503115,17792.733546
2,test,m_pid,468.089081,1.408079,14673.570616
3,test,ta_pid,10.940620,7.722339,8623.413930
